# Семинар 4 - Ассемблер x86

*Дисклемйер: в этой теме важна архитектура компьютера, на котором вы работаете. Например, если у вас Mac на Apple Silicon, примеры работать не будут. Чтобы это обойти, вы можете использовать виртуалки, например, [lima](https://lima-vm.io) с qemu движком* 

Один из способов изучения ассемблера - посмотреть на то, какой код на нем генерирует компилятор. В частности, можно дизассемблировать даже бинарные файлы. Скомпилируем простую программу из `sum.c` и посмотрим на вывод дизассемблера

In [16]:
!cat snippets/c-sum/sum.c 

int main () {
    int a = 2;
    int b = 3;
    int c = a + b;

    return 0;
}

In [ ]:
#!objdump -M intel intel-mnemonic -d snippets/c-sum/sum.out

Можно попросить компилятор выдать ассемблер по исходному коду:

In [17]:
#!gcc snippets/c-sum/sum.c -S -o snippets/c-sum/sum.s -masm=intel
!cat snippets/c-sum/sum.s

	.file	"sum.c"
	.intel_syntax noprefix
	.text
	.globl	main
	.type	main, @function
main:
.LFB0:
	.cfi_startproc
	endbr64
	push	rbp
	.cfi_def_cfa_offset 16
	.cfi_offset 6, -16
	mov	rbp, rsp
	.cfi_def_cfa_register 6
	mov	DWORD PTR -12[rbp], 2
	mov	DWORD PTR -8[rbp], 3
	mov	edx, DWORD PTR -12[rbp]
	mov	eax, DWORD PTR -8[rbp]
	add	eax, edx
	mov	DWORD PTR -4[rbp], eax
	mov	eax, 0
	pop	rbp
	.cfi_def_cfa 7, 8
	ret
	.cfi_endproc
.LFE0:
	.size	main, .-main
	.ident	"GCC: (Ubuntu 14.2.0-19ubuntu2) 14.2.0"
	.section	.note.GNU-stack,"",@progbits
	.section	.note.gnu.property,"a"
	.align 8
	.long	1f - 0f
	.long	4f - 1f
	.long	5
0:
	.string	"GNU"
1:
	.align 8
	.long	0xc0000002
	.long	3f - 2f
2:
	.long	0x3
3:
	.align 8
4:


> Удобный инструмент для анализа вывода компиляторов [Godbolt](https://godbolt.org/)

Можно заметить, что программа на ассемблере состоит из каких-то команд. Они кодируются разным количество байт, то есть x86_64 является `CISC (complex instruction set computing)` архитектурой. Аргументы вида `rax`, `eax` и подобные - это имена регистров, то есть ячеек памяти процессора. Квадратные скобки означают обращения по адресу в оперативной памяти (аналог разименования указателей).

### Регистры и соглашения о вызовах


В `x86_64` предусмотрены следующие соглашения об использовании регистров:

| Регистры        | Назначение                                                   |
| --------------- | ------------------------------------------------------------ |
| `rax`   | возвращаемое значение функции         |
| `rdi, rsi, rdx, rcx, r8, r9` | аргументы функции, последующие кладутся на стек |
| `rax, rdi, rsi, rdx, rcx, r8, r9, r10, r11`  | временные регистры, для которые не гарантируется сохранение результата, если вызывать какую-либо функцию |
| `rbx, rsp, rbp, r12, r13, r14, r15` | регистры, для которых гарантируется, что вызываемая функция их не будет портить |
| `rbp`           | указатель на границу фрейма функции, обычно используется отладчиком |
| `rsp`      | указатель на вершину стека                                   |

Обращаться к младшим 32 битам можно заменяя `r` на `e` (например `eax`) в начале имени регистра или дописывая `d` в случае номерных (например `r13d`).

`mov register [base + index * scale + offset]` позволяет загрузить значение в регистр из памяти. Также можно записать значение в память из регистра или из регистра в регистр. Обратите внимание, что такой синтаксис крайне удобен для работы с массивами (пример использования можно посмотреть в `arr_get.S`).

Для загрузки в регистры чисел меньшей размерности служат суффиксы. Например, `movsxd` позволяет загружать знаковые 32-битные числа.

### Арифметические команды

Общий вид команд: `CMD left, right` отвечает за операцию `left*=right`. Например: `ADD r8, r9` прибавляет к значению регистра `r8` значения регистра `r9`.

```
add     DST, SRC        /* DST += SRC */
sub     DST, SRC        /* DST -= SRC */
inc     DST             /* ++DST */
dec     DST             /* --DST */
neg     DST             /* DST = -DST */
mov     DST, SRC        /* DST = SRC */
imul    SRC             /* (eax,edx) = eax * SRC - знаковое */
mul     SRC             /* (eax,edx) = eax * SRC - беззнаковое */
and     DST, SRC        /* DST &= SRC */
or      DST, SRC        /* DST |= SRC */
xor     DST, SRC        /* DST ^= SRC */
not     DST             /* DST = ~DST */
cmp     DST, SRC        /* DST - SRC, результат не сохраняется, */
test    DST, SRC        /* DST & SRC, результат не сохраняется  */
adc     DST, SRC        /* DST += SRC + CF */
sbb     DST, SRC        /* DST -= SRC - CF */
```

### Метки и переходы

Можно создавать метки и переходить по ним. Для безусловного перехода служит команда `jmp`. Для сравнения используется команда `cmp`. Она выставляет флаги, результаты которых можно использовать в командах для условных переходов:

```
jz      label   /* переход, если равно (нуль), ZF == 1 */
jnz     label   /* переход, если не равно (не нуль), ZF == 0 */
jc      label   /* переход, если CF == 1 */
jnc     label   /* переход, если CF == 0 */
jo      label   /* переход, если OF == 1 */
jno     label   /* переход, если OF == 0 */
jg      label   /* переход, если больше для знаковых чисел */
jge     label   /* переход, если >= для знаковых чисел */
jl      label   /* переход, если < для знаковых чисел */
jle     label   /* переход, если <= для знаковых чисел */
ja      label   /* переход, если > для беззнаковых чисел */
jae     label   /* переход, если >= (беззнаковый) */
jb      label   /* переход, если < (беззнаковый) */
jbe     label   /* переход, если <= (беззнаковый) */
```

### Стек и вызов функций

<img src="media/stack_x64.png" alt="stack layout" width="400" style="background-color:white;"/> <img src="media/stack.png" alt="stack layout" width="200" style="background-color:white;"/>

Переход по метке функции осуществляется командой `call label_name`, которая кладёт адрес возврата на стек, подменяет его на следующую команду после `call` и передаёт управление функции. Для выхода из функции служит команда `ret`. Пример использования функций стандартной библиотеки можно найти в 

На картинке можно видеть, что после адреса возврата лежит RBP. Во времена 32-битных систем стандартом было поддеживать текущий stack frame. Первое, что делалось внутри функции при вызове -- сохранение прошлого начала стека. Следующая конструкция называлась прологом:

```
push rbp
mov rbp, rsp
```

В конце функции принят был эпилог

``
mov rsp, rbp
pop rbp
``

Сейчас это необязательно, потому что компилятор в состоянии высчитать все адреса от вершины стека. Но вам самим в коде может быть удобно адресовать локальные переменне от RBP, потому что RSP менятся.

### Попробуем пописать на ассемблере
Сделаем функцию, которая принимает 2 числа и возвращает их сумму

In [25]:
!cat snippets/asm-sum/sum_func.s 

.intel_syntax noprefix
.global sum

sum:
    mov rax, 0
    add rax, rdi
    add rax, rsi
    ret


In [26]:
!cat snippets/asm-sum/sum_func.c

#include <stdio.h>

int sum(int a, int b);

int main() {
    int a = 3;
    int b = 2;
    int c = sum(a, b);
    printf("%d\n", c);
}


In [24]:
#!gcc snippets/asm-sum/sum_func.c snippets/asm-sum/sum_func.s -o snippets/asm-sum/sum_func.out
!./snippets/asm-sum/sum_func.out

zsh:1: exec format error: ./snippets/asm-sum/sum_func.out


### Попробуем теперь написать вычисление чисел Фибоначчи разными способами. Сначала через цикл:

In [30]:
!cat snippets/fib-loop/fib.s

.intel_syntax noprefix
.global fib

fib:
    mov rax, 0
    mov r8, 1

.loop:
    cmp rdi, 0
    jle .exit
    dec rdi

    mov r9, rax 
    mov rax, r8 
    add r8, r9

    jmp .loop

.exit:
    ret 



In [31]:
#!gcc snippets/fib-loop/fib.c snippets/fib-loop/fib.s -o snippets/fib-loop/fib.out
!./snippets/fib-loop/fib.out

zsh:1: exec format error: ./snippets/fib-loop/fib.out


#### И рекурсивный вариант:

In [32]:
!cat snippets/fib-rec/fib.s

.intel_syntax noprefix
.global fib

fib:
    push r12
    push r13

    mov rax, 0
    cmp rdi, 2 
    jl .return
    je .return1

    dec rdi
    mov r12, rdi

    call fib 
    mov r13, rax 
    
    mov rdi, r12
    dec rdi
    call fib 
    add rax, r13

.return:
    pop r13
    pop r12
    ret 

.return1:
    pop r13
    pop r12
    mov rax, 1
    ret


In [33]:
#!gcc snippets/fib-rec/fib.c snippets/fib-rec/fib.s -o snippets/fib-rec/fib.out
!./snippets/fib-rec/fib.out

zsh:1: exec format error: ./snippets/fib-rec/fib.out


### Hello, world на ассемблере:

In [34]:
!cat snippets/hello-world/hello.s

.intel_syntax noprefix
.section .data
    hello: .asciz "Hello, world!\n"

.section .text
    .global _start

_start:
    # Write system call
    mov rax, 1          # syscall number for write (1)
    mov rdi, 1          # file descriptor stdout (1)
    lea rsi, [hello]    # pointer to string (use lea for address loading)
    mov rdx, 14         # string length
    syscall

    # Exit system call
    mov rax, 60         # syscall number for exit (60)
    mov rdi, 0          # exit status 0
    syscall


In [35]:
#!gcc snippets/hello-world/hello.s -o snippets/hello-world/hello.out -nostdlib -static
!./snippets/hello-world/hello.out

zsh:1: exec format error: ./snippets/hello-world/hello.out


```bash
strace ./hello.out

execve("./hello.out", ["./hello.out"], 0x7ffe3e11f9c0 /* 24 vars */) = 0
write(1, "Hello, world!\n", 14)         = 14
exit(0)                                 = ?
+++ exited with 0 +++
```

### Попробуем вызвать С функцию из ассемблера:

In [36]:
!cat snippets/ld/hello.s

.intel_syntax noprefix
.section .data
    hello: .asciz "Hello, world!\n"

.section .text
    .global _start

_start:
    lea rdi, [hello]
    call printf
    
    # Exit properly
    mov rax, 60                # sys_exit
    xor rdi, rdi               # exit status 0
    syscall




In [43]:
#!as -o snippets/ld/hello.o snippets/ld/hello.s
#!ld -o snippets/ld/hello.out snippets/ld/hello.o -lc -dynamic-linker /lib64/ld-linux-x86-64.so.2 -e _start

In [44]:
#!snippets/ld/hello.out

### Поработаем с памятью
Даны n, x. Посчитаем $\sum_{i=0}^{n - 1} (-1)^i \cdot x[i]$

In [48]:
!cat snippets/memory/sum.s

.intel_syntax noprefix
.text
.globl my_sum
my_sum:
    // n - edi
    // x - rsi
    mov eax, 0
    cmp edi, 0
start_loop:   
    jle return_eax
    add eax, DWORD PTR [rsi]
    add rsi, 4
    dec edi # and write compare with 0 flags
    
    jle return_eax
    sub eax, DWORD PTR [rsi]
    add rsi, 4
    dec edi # and write compare with 0 flags
    
    jmp start_loop
return_eax:
    ret


In [49]:
!cat snippets/memory/sum.c

#include <stdint.h>
#include <stdio.h>
#include <assert.h>
    
int32_t my_sum(int32_t n, int32_t* x);

int main() {
    int32_t x[] = {100, 2, 200, 3};
    assert(my_sum(sizeof(x) / sizeof(int32_t), x) == 100 - 2 + 200 - 3);
    int32_t y[] = {100, 2, 200};
    assert(my_sum(sizeof(y) / sizeof(int32_t), y) == 100 - 2 + 200);
    int32_t z[] = {100};
    assert(my_sum(sizeof(z) / sizeof(int32_t), z) == 100);
    printf("SUCCESS");
    return 0;
}

In [50]:
#!gcc snippets/memory/sum.c snippets/memory/sum.s -o snippets/memory/sum.out
#!snippets/memory/sum.out

## Развлекательно-познавательная часть

In [51]:
!cat snippets/div/div.c

#include <stdint.h>
    
int32_t div(int32_t a) { 
    return a / 4;
}

uint32_t udiv(uint32_t a) { 
    return a / 4;
}

```asm
div:
        lea     eax, [rdi + 3]
        test    edi, edi
        cmovns  eax, edi
        sar     eax, 2
        ret

udiv:
        mov     eax, edi
        shr     eax, 2
        ret
```

> [C Inline asm](https://gcc.gnu.org/onlinedocs/gcc/Using-Assembly-Language-with-C.html)

## ELF файлы

ELF (Executable and Linking Format) - формат для исполняемых файлов в Linux

<img src="media/elf.jpg" alt="elf file" width="400" style="background-color:white;"/>

```bash
readelf -h /bin/ls | grep Magic
  Magic:   7f 45 4c 46 02 01 01 00 00 00 00 00 00 00 00 00
```

Секции ELF файлов:
* `.text` - секция с кодом
* `.data` - глобальные переменные 
* `.bss` - неинициализированные данные, которые должно быть заполнены нулями. В самом файле на диске эта секция остутствует, загрузчик заполняет ее нулями при запуске
* `.rodata` - read-only данные, например, строковые константы
* `.comment`, `.note` - комментарии, оставленные компилятором/линкером
* `.stab`, `.stabstr` - дебажные символы

## SIMD, AVX, SSE

В 80-е годы операции с вещественным числами исполнялись на отдельном сопроцессоре x87. У него был свой стек и набор инструкций. Сейчас же вычисления с плавающей точкой выполняются самим процессором над регистрами семейств SSE (128 бит)/AVX (256 бит)/AVX-512 (512 бит). Называются они `xmm0-15`, `ymm` и `zmm` соответственно. `xmm0` соответсвует младшим 128 битам `ymm0` и так далее.

[MMX](https://ru.wikipedia.org/wiki/MMX) (1997) (Multimedia Extensions — мультимедийные расширения) 

64-битные регистры mm0..mm7 (устарело)

[SSE](https://ru.wikipedia.org/wiki/SSE) (1999) (Streaming SIMD Extensions)

128-битные регистры xmm0..xmm7 (связи с регистрами MMX нет вроде бы) (количество может быть 16 и 32 на новых процессорах)

[AVX](https://ru.wikipedia.org/wiki/AVX) (2008) (Advanced Vector Extensions)

256-битные регистры ymm0 — ymm15 (регистры SSE становятся младшими половинками регистров AVX) (количество может быть 32 на новых процессорах)

[AVX-512](https://en.wikipedia.org/wiki/AVX-512) (2013) (Advanced Vector Extensions 512 bits)

512-битные регистры zmm0-zmm31 (регистры AVX становятся младшими половинками регистров AVX-512)

### Скалярные инструкции
Векторные регистры можно использовать для операций с одиночными вещественными числами. Команды похожи на обычные арифметические. Буква `s` на конце означает, что операция над `float`, буква `d`, что над `double`. Есть также операции для конвертации и сравнения.

```
// Копирование регистр-регистр и регистр-память
movsd   DST, SRC  // пересылка double
movss   DST, SRC  // пересылка float

// Арифметические
addsd   DST, SRC   // DST += SRC, double
addss   DST, SRC   // DST += SRC, float
subsd   DST, SRC   // DST -= SRC, double
subss   DST, SRC   // DST -= SRC, float
mulsd   DST, SRC   // DST *= SRC, double
mulss   DST, SRC   // DST *= SRC, float
divsd   DST, SRC   // DST /= SRC, double
divss   DST, SRC   // DST /= SRC, float
sqrtsd  DST, SRC   // DST = sqrt(SRC), double
sqrtss  DST, SRC   // DST = sqrt(SRC), float
maxsd   DST, SRC   // DST = max(DST, SRC), double
maxss   DST, SRC   // DST = max(DST, SRC), float
minsd   DST, SRC   // DST = min(DST, SRC), double
minss   DST, SRC   // DST = min(DST, SRC), float

// Преобразования
cvtsd2si DST, SRC  // double -> int
cvtsi2sd DST, SRC  // int -> double

// Сравнения (операция DST-SRC, которая меняет флаги)
comisd  DST, SRC  // для double
comiss  DST, SRC  // для float
```

Вещественные аргументы в функцию передаются в `xmm0-xmm7`, значение возвращается в `xmm0`.

### Векторные инструкции

Также можно оперировать с векторными регистрами как с массивом из нескольких чисел. В случае AVX можно, например, сложить 8 вещественных чисел за одну операцию и, таким образом, получить кратный прирост производительности.

Команды имеют вид `v(operation_name)p[s|d]`:

`v` означает, что команда оперирует векторным регистром
`p` означает, что в нём упаковано несколько значений
`s` означает `float`, `d` означает double
Например: `vaddps dst, src1, src2`.

Для загрузки значений служит: `vmov[a|u]p[s|d]`. `a` означает, что память выровнена по размеру векторного регистра (выделить такую память можно, например, с помощью `aligned_alloc`).

Пример сложения 2 векторов с помощью AVX:

In [60]:
!cat snippets/avx-sum/add.S

    .global very_important_function
    .intel_syntax noprefix
    .text

// (N=rdi, *A=rsi, *B=rdx, *R=rcx)
very_important_function:
    
    mov r8, rdi

    .Loop:
        sub rdi, 8
        
        vmovaps ymm0, [rsi + rdi * 4]
        vmovaps ymm1, [rdx + rdi * 4]
        vaddps ymm0, ymm0, ymm1
        vmovaps [rcx + rdi * 4], ymm0
        
        cmp rdi, 0
        jg .Loop

    ret


In [61]:
!cat snippets/avx-sum/add.c

#include <limits.h>
#include <stdalign.h>
#include <stdio.h>
#include <stdlib.h>

extern double
very_important_function(int N, const float *A, const float *B, float *R);

int main(){
	const size_t n = 8;
	const size_t alig = alignof(float) * CHAR_BIT;
	const size_t sz = alig * n;
	float *a = aligned_alloc(alig , sz);
	float *b = aligned_alloc(alig, sz);
	float *c = aligned_alloc(alig, sz);
	for (int i = 0; i < n; i++){
		a[i] = i + 1;
		b[i] = 2 * (i + 1);
	}
	double res = (float)very_important_function(8, a, b, c);
	for (int i = 0; i < n; i++) {
		printf("%f ", c[i]);
	}
	printf("\n");
	return 0;
}


In [64]:
#!gcc snippets/avx-sum/add.S snippets/avx-sum/add.c -o snippets/avx-sum/add.out

### Интринсики

Чтобы использовать векторные инструкции в программах на `C`, есть специальные обёртки, называемые `intrinsics`.

```
for (int i = 0; i < n; i+= 8) {
    __m256 r1 = _mm256_load_ps(a + i);
    __m256 r2 = _mm256_load_ps(b + i);
    __m256 r3 = _mm256_add_ps(r1, r2);
    _mm256_store_ps(&c[i], r3);
}
```

Здесь `__m256` на самом деле не переменная. Этот код развернётся в загрузку значения в регистр (можно проверить с помощью Godbolt).

Справочник по `intrinsics`: [Intel Intrinsics Guide](https://www.laruence.com/sse/#).


Пример использования интринсиков в C коде:

In [65]:
!cat snippets/intrinsics/add.c

#include <limits.h>
#include <stdalign.h>
#include <stdio.h>
#include <stdlib.h>

#include <immintrin.h>

int main(){
	const size_t n = 8;
	const size_t alig = alignof(float) * CHAR_BIT;
	const size_t sz = alig * n;
	float *a = aligned_alloc(alig , sz);
	float *b = aligned_alloc(alig, sz);
	float *c = aligned_alloc(alig, sz);
	for (int i = 0; i < n; i++){
		a[i] = i + 1;
		b[i] = 2 * (i + 1);
	}
	for (int i = 0; i < n; i+= 8) {
        __m256 r1 = _mm256_load_ps(a + i);
        __m256 r2 = _mm256_load_ps(b + i);
        __m256 r3 = _mm256_add_ps(r1, r2);
        _mm256_store_ps(&c[i], r3);
    }
	for (int i = 0; i < n; i++) {
		printf("%f ", c[i]);
	}
	printf("\n");
	return 0;
}


In [66]:
#!gcc snippets/intrinsics/add.c -mavx -o snippets/intrinsics/add.out
#!./snippets/intrinsics/add.c